In [1]:
!pip install -q kaggle

In [2]:
import os
from getpass import getpass

os.environ["KAGGLE_API_TOKEN"] = getpass("KGAT_1c9f6c4bbf373df37be2179372fb2751")

KGAT_1c9f6c4bbf373df37be2179372fb2751··········


In [3]:
!kaggle datasets download -d grassknoted/asl-alphabet -p /content/asl --unzip

Dataset URL: https://www.kaggle.com/datasets/grassknoted/asl-alphabet
License(s): GPL-2.0
100% 1.03G/1.03G [00:27<00:00, 40.2MB/s]



In [4]:
from pathlib import Path
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, random_split
import torch

raw = Path("/content/asl/asl_alphabet_train/asl_alphabet_train")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = datasets.ImageFolder(raw, transform=transform)

# Select 200 images per class
indices = []
for class_index in range(len(dataset.classes)):
    class_indices = [
        i for i, (_, label) in enumerate(dataset.samples)
        if label == class_index
    ]
    indices.extend(class_indices[:200])

small_dataset = Subset(dataset, indices)

train_size = int(0.8 * len(small_dataset))
val_size = len(small_dataset) - train_size

train_dataset, val_dataset = random_split(
    small_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

num_classes = len(dataset.classes)

print("Number of classes:", num_classes)
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))

Number of classes: 29
Training images: 4640
Validation images: 1160


In [5]:
import torch
import torch.nn as nn
import time

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes=29

In [6]:
class ASLCNN(nn.Module):
    def __init__(self, num_classes=29):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Reduce each feature map to 1 × 1
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [7]:
model=ASLCNN(num_classes).to(device)
print(model)

ASLCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): AdaptiveAvgPool2d(output_size=(1, 1))
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=128, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=256, out_features=29, bias=True)
  )
)


Trainable Params

In [8]:
total_params=sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)
print(total_params)

133725


Loss and optimizer

In [9]:
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)

Train CNN

In [11]:
epochs=25
start_time=time.time()

for epoch in  range(epochs):
  model.train()

  total_loss=0
  correct=0
  total=0

  for images,labels in train_loader:
    images=images.to(device)
    labels=labels.to(device)

    #Feed Forward
    outputs=model(images)

    #Loss calculation
    loss=criterion(outputs,labels)

    #Clear gradient
    optimizer.zero_grad()

    #Backpropagation
    loss.backward()

    #Update weights
    optimizer.step()

    total_loss += loss.item() * images.size(0)
    predictions = outputs.argmax(dim=1)
    correct += (predictions == labels).sum().item()
    total += labels.size(0)

  epoch_loss = total_loss / total
  epoch_accuracy = 100 * correct / total

  print(
        f"Epoch [{epoch + 1}/{epochs}] "
        f"Loss: {epoch_loss:.4f} "
        f"Accuracy: {epoch_accuracy:.2f}%"
    )

training_time = time.time() - start_time
print(f"Training time: {training_time:.2f} seconds")

Epoch [1/25] Loss: 1.3508 Accuracy: 49.12%
Epoch [2/25] Loss: 1.2506 Accuracy: 52.89%
Epoch [3/25] Loss: 1.0121 Accuracy: 61.92%
Epoch [4/25] Loss: 0.8962 Accuracy: 66.53%
Epoch [5/25] Loss: 0.8038 Accuracy: 70.06%
Epoch [6/25] Loss: 0.7165 Accuracy: 73.45%
Epoch [7/25] Loss: 0.6524 Accuracy: 75.04%
Epoch [8/25] Loss: 0.5996 Accuracy: 77.61%
Epoch [9/25] Loss: 0.5662 Accuracy: 78.99%
Epoch [10/25] Loss: 0.5521 Accuracy: 79.91%
Epoch [11/25] Loss: 0.4819 Accuracy: 82.56%
Epoch [12/25] Loss: 0.4619 Accuracy: 83.49%
Epoch [13/25] Loss: 0.4236 Accuracy: 84.70%
Epoch [14/25] Loss: 0.3803 Accuracy: 86.38%
Epoch [15/25] Loss: 0.3835 Accuracy: 86.10%
Epoch [16/25] Loss: 0.3892 Accuracy: 86.77%
Epoch [17/25] Loss: 0.3058 Accuracy: 89.14%
Epoch [18/25] Loss: 0.2956 Accuracy: 89.85%
Epoch [19/25] Loss: 0.2871 Accuracy: 89.66%
Epoch [20/25] Loss: 0.2579 Accuracy: 91.19%
Epoch [21/25] Loss: 0.2180 Accuracy: 92.59%
Epoch [22/25] Loss: 0.2424 Accuracy: 91.77%
Epoch [23/25] Loss: 0.2326 Accuracy: 91.9

Evaluate Model

In [12]:
# Evaluate the custom CNN
model.eval()

val_loss = 0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        val_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

val_loss = val_loss / total
val_accuracy = 100 * correct / total

print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.2f}%")

Validation Loss: 0.1724
Validation Accuracy: 94.66%


Saving CNN

In [13]:
from pathlib import Path

Path("/content/results").mkdir(parents=True, exist_ok=True)

torch.save({
    "model_state_dict": model.state_dict(),
    "classes": dataset.classes
}, "/content/results/custom_CNN.pth")

print("ResNet18 model saved!")

ResNet18 model saved!


In [14]:
from google.colab import files

files.download("/content/results/custom_CNN.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>